# Phase 2 · Step 2 — Embed chunks, upsert into Qdrant

Re-loads and re-chunks the sample PDF the same way
`01_pdf_ingestion.ipynb` did, embeds each chunk with `fastembed`'s
`BAAI/bge-small-en-v1.5`, and upserts them as points into the
`papermind_chunks` collection created in `02_qdrant_setup.ipynb`. See
`learning/04_rag_pipeline/02_embed_upsert.md` for the why behind these
choices.

## Reload + re-chunk

Same loading/chunking code as notebook 01 — duplicated inline since there's
no shared module yet.

In [1]:
from generate_sample_pdf import generate
from llama_index.core.node_parser import SentenceSplitter
from llama_index.readers.file import PDFReader

pdf_path = generate()
documents = PDFReader().load_data(file=pdf_path)
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

print(f"Produced {len(nodes)} chunks from {len(documents)} pages")

Produced 4 chunks from 4 pages


## Connect to the existing collection

Same client/constants as notebook 02. We only connect here — the collection
already exists from Step 1, so we assert that instead of recreating it.

In [2]:
from pathlib import Path

from qdrant_client import QdrantClient

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
COLLECTION_NAME = "papermind_chunks"
STORAGE_PATH = Path("data/qdrant")

client = QdrantClient(path=str(STORAGE_PATH))
assert client.collection_exists(COLLECTION_NAME)
client

## Embed

`TextEmbedding(...).embed(texts)` returns one 384-dim vector per input
string, in the same order as the input list.

In [3]:
from fastembed import TextEmbedding

embedder = TextEmbedding(model_name=EMBEDDING_MODEL)
texts = [node.get_content() for node in nodes]
vectors = list(embedder.embed(texts))

print(f"Embedded {len(vectors)} chunks, dim={len(vectors[0])}")

Embedded 4 chunks, dim=384


## Build point IDs + payloads

Each point's ID is a deterministic `uuid5` built from
`file_name:page_label:chunk_index` (the counter resets per page), so
re-running this notebook overwrites the same points instead of duplicating
them.

In [4]:
from uuid import NAMESPACE_URL, uuid5

from qdrant_client.models import PointStruct

page_counters: dict[str, int] = {}
points = []

for node, vector in zip(nodes, vectors):
    page_label = node.metadata["page_label"]
    file_name = node.metadata["file_name"]
    chunk_index = page_counters.get(page_label, 0)
    page_counters[page_label] = chunk_index + 1

    point_id = str(uuid5(NAMESPACE_URL, f"{file_name}:{page_label}:{chunk_index}"))
    points.append(
        PointStruct(
            id=point_id,
            vector=vector.tolist(),
            payload={
                "text": node.get_content(),
                "page_label": page_label,
                "file_name": file_name,
                "chunk_index": chunk_index,
            },
        )
    )

print(f"Built {len(points)} points")

Built 4 points


## Upsert

In [5]:
client.upsert(collection_name=COLLECTION_NAME, points=points)
print("Upsert complete")

Upsert complete


## Verify

No search here (that's Step 3) — just confirm the count matches and one
point's payload/vector look right.

In [6]:
info = client.get_collection(COLLECTION_NAME)
print("Points count:", info.points_count)
assert info.points_count == len(nodes)

sample = client.retrieve(collection_name=COLLECTION_NAME, ids=[points[0].id], with_vectors=True)[0]
print("Sample id:", sample.id)
print("Sample payload:", sample.payload)
print("Vector length:", len(sample.vector))

Points count: 4
Sample id: c94c7d3f-ee2e-5d31-b61b-5c39dab07f0c
Sample payload: {'text': 'Abstract\nThis paper studies retrieval-augmented generation for question answering over research\npapers. We show that combining dense retrieval with a re-ranking stage improves citation\naccuracy compared to retrieval alone.', 'page_label': '1', 'file_name': 'sample.pdf', 'chunk_index': 0}
Vector length: 384


## Next

Phase 2 · Step 3 runs dense vector search against this now-populated
collection, embedding a query the same way and returning top-k chunks with
their `page_label` as citations.